# Perceptron from Scratch — Banknote AuthenticationThis notebook applies **Rosenblatt's Perceptron** (1958) — implemented from scratch in our `mlpkg` package — to the **UCI Banknote Authentication** dataset. The task is to classify banknotes as genuine or forged based on statistics extracted from wavelet transforms of their images.This is a perfect dataset for the perceptron because it is **(approximately) linearly separable** — exactly the regime where the Perceptron Convergence Theorem guarantees the algorithm will find a separating hyperplane.**The perceptron update rule** (only fires on errors):$$w \leftarrow w + \eta \cdot y_i \cdot x_i,\quad b \leftarrow b + \eta \cdot y_i$$**Dataset:** 1372 banknote images. 4 features: variance, skewness, kurtosis, and entropy of the wavelet-transformed image. Source: [UCI](https://archive.ics.uci.edu/dataset/267/banknote+authentication).

In [ ]:
import osimport urllib.requestDATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data") if os.path.basename(os.getcwd()) == "notebooks" else "data"os.makedirs(DATA_DIR, exist_ok=True)def download_if_needed(url, filename):    """Download a CSV if it doesn't already exist locally."""    path = os.path.join(DATA_DIR, filename)    if not os.path.exists(path):        print(f"Downloading {filename} from {url}")        urllib.request.urlretrieve(url, path)    return pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import accuracy_score, confusion_matrixfrom sklearn.linear_model import Perceptron as SklearnPerceptron# Our from-scratch implementation:from mlpkg.supervised import PerceptronScratchsns.set_style("whitegrid")np.random.seed(42)

## 1. Load data

In [ ]:
path = download_if_needed(    "https://raw.githubusercontent.com/jbrownlee/Datasets/master/banknote_authentication.csv",    "banknote.csv",)cols = ["variance", "skewness", "kurtosis", "entropy", "class"]df = pd.read_csv(path, header=None, names=cols)print("Shape:", df.shape)print("Class balance:", df["class"].value_counts().to_dict())df.head()

## 2. Visualize separability

In [ ]:
sns.pairplot(df, hue="class", height=1.6, plot_kws={"alpha": 0.6, "s": 12})plt.suptitle("Banknote features — class structure", y=1.02)plt.show()

Notice how the classes form mostly separable clusters — ideal for a linear classifier like the perceptron.

## 3. Split + scale

In [ ]:
X = df.drop(columns="class").valuesy = df["class"].valuesX_tr, X_te, y_tr, y_te = train_test_split(    X, y, test_size=0.25, random_state=42, stratify=y)scaler = StandardScaler()X_tr_s = scaler.fit_transform(X_tr)X_te_s = scaler.transform(X_te)

## 4. Fit our from-scratch perceptron

In [ ]:
ours = PerceptronScratch(learning_rate=0.01, n_epochs=100, random_state=42)ours.fit(X_tr_s, y_tr)print(f"Training set accuracy: {ours.score(X_tr_s, y_tr):.4f}")print(f"Test set accuracy:     {ours.score(X_te_s, y_te):.4f}")print(f"Converged in {len(ours.errors_per_epoch_)} epochs")

## 5. Convergence plot

In [ ]:
plt.figure(figsize=(8, 4))plt.plot(range(1, len(ours.errors_per_epoch_) + 1),         ours.errors_per_epoch_, marker="o")plt.xlabel("Epoch"); plt.ylabel("Misclassifications")plt.title("Perceptron convergence — errors decrease over epochs")plt.tight_layout(); plt.show()

If the data is linearly separable, the perceptron is *guaranteed* to converge to zero errors in finite time (Perceptron Convergence Theorem). On real-world data with overlap it may oscillate but still find a strong solution.

## 6. Compare to sklearn's Perceptron

In [ ]:
ref = SklearnPerceptron(eta0=0.01, max_iter=100,                       random_state=42, tol=None)ref.fit(X_tr_s, y_tr)print(f"mlpkg PerceptronScratch test accuracy: {ours.score(X_te_s, y_te):.4f}")print(f"sklearn Perceptron test accuracy:      {ref.score(X_te_s, y_te):.4f}")

## 7. Confusion matrix

In [ ]:
cm = confusion_matrix(y_te, ours.predict(X_te_s))sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",            xticklabels=["genuine", "forged"],            yticklabels=["genuine", "forged"])plt.xlabel("Predicted"); plt.ylabel("Actual")plt.title("Perceptron — Confusion Matrix")plt.tight_layout(); plt.show()

## Takeaways- Our from-scratch perceptron achieves >97% test accuracy — comparable to sklearn's reference implementation.- The convergence curve drops rapidly: most learning happens in the first 5–10 epochs.- The perceptron's strength is its simplicity. Its limitation: it can only learn **linearly separable** decision boundaries. The XOR problem famously broke it (Minsky & Papert, 1969), motivating the development of multi-layer networks (covered in lecture 6).- Feature scaling is helpful but not required for the perceptron — the algorithm works on raw features too, just with potentially slower convergence.